# Pathway 2: Multi-Solid Global Assembly

This tutorial demonstrates assembling multiple geometric parts into a single domain and solving
the fused system globally.

```
EMProject  →  Create Assembly  →  Add Components  →  Build & Mesh  →  FDS Solve  →  Plot & Compare
```

## Overview

When multiple parts are **fused** into a single domain via `build()`,
the interfaces between them disappear and the solver sees a single continuous mesh.
This pathway is ideal for small assemblies where solving the fused system
is computationally feasible.

## 1. Create Project and Assembly

The `EMProject` manages the full simulation lifecycle.
An `Assembly` groups multiple parts along a principal axis.

In [ ]:
from core.em_project import EMProject
from geometry.primitives import RectangularWaveguide
import matplotlib.pyplot as plt
%matplotlib widget

# 1. Create the project
project_name = 'pathway2_global_assembly'
base_dir = './simulations'  # Change to your preferred simulation directory
proj = EMProject(name=project_name, base_dir=base_dir)

## 2. Add Components

We add two rectangular waveguide sections.
The `after` argument auto-aligns the second component flush against the first.

In [ ]:
# 2. Create assembly and add components
#    Uncomment the following lines on first run.
#    On subsequent runs, the project loads automatically.

# assembly = proj.create_assembly(main_axis='Z')
# wg1 = RectangularWaveguide(a=0.1, L=0.1)
# assembly.add("rwg1", wg1)
# assembly.add("rwg1", wg1, after="rwg1")
# assembly.build()
# assembly.generate_mesh(maxh=0.02)

# Visualise the mesh
proj.geo.show('mesh')

## 3. Solve the Full-Order Model (FOM)

Since the assembly is a **compound** structure (multiple domains), the solver
automatically performs per-domain solving and concatenation.

The solver is configured using a config dictionary:

| Parameter | Description |
|-----------|-------------|
| `nportmodes` | Number of modes per port |
| `order` | Polynomial order of the finite element space |
| `nsamples` | Number of frequency sample points |
| `fmin` | Minimum frequency (GHz) |
| `fmax` | Maximum frequency (GHz) |
| `solver_type` | `'direct'` or `'iterative'` |

In [ ]:
# 3. Configure and run the frequency domain solver
fom_config = {
    'nportmodes': 3,
    'order': 3,
    'nsamples': 100,
    'fmin': 1e-3,
    'fmax': 5,
    'solver_type': 'direct',
    # 'rerun': True  # Uncomment to force re-solve even if results exist
}

fom_result = proj.fds.solve(config=fom_config)

## 4. Plot S-Parameters

For a **compound** structure (multiple domains), results are accessed via `proj.fds.foms`
(note the plural). The parameter labels use the format `'port(mode)port(mode)'`.

In [ ]:
# 4. Plot S-parameters: magnitude and phase
which = [['1(1)1(1)'], ['1(1)2(1)']]

fig, axs = plt.subplot_mosaic(
    [[1, 2], [3, 4]], figsize=(10, 8), layout='constrained'
)

for idx, wh in enumerate(which):
    # Magnitude
    proj.fds.foms.plot_s(wh, ax=axs[idx + 1])
    # Phase
    proj.fds.foms.plot_s(wh, plot_type='phase', ax=axs[idx + 3])

fig.suptitle('Global Assembly — S-Parameters', fontsize=14)

## 5. Compare with Analytical Solution

The `RWGAnalytical` class provides closed-form S- and Z-parameters for a
rectangular waveguide. Note that the total length is the sum of the two
components: $L = 0.1 + 0.1 = 0.2$ m.

In [ ]:
from analytical.rectangular_waveguide import RWGAnalytical

# Create analytical model with the total length (sum of both components)
analytical = RWGAnalytical(
    a=0.1, L=0.2,
    freq_range=(fom_config['fmin'], fom_config['fmax'])
)

# Plot comparison: FOM vs Analytical
which = [['1(1)1(1)'], ['1(1)2(1)']]

fig, axs = plt.subplot_mosaic(
    [[1, 2], [3, 4]], figsize=(10, 8), layout='constrained'
)

for idx, wh in enumerate(which):
    # Magnitude
    analytical.plot_s(wh, ax=axs[idx + 1])
    proj.fds.foms.plot_s(wh, ax=axs[idx + 1])
    # Phase
    analytical.plot_s(wh, plot_type='phase', ax=axs[idx + 3])
    proj.fds.foms.plot_s(wh, plot_type='phase', ax=axs[idx + 3])

fig.suptitle('FOM vs Analytical — S-Parameters', fontsize=14)

## Summary

In this tutorial we:
1. Created an `EMProject` and a multi-component assembly with automatic alignment
2. Built and meshed the fused assembly
3. Configured and ran the Frequency Domain Solver via the `config` dictionary
4. Plotted compound S-parameter magnitude and phase using `proj.fds.foms`
5. Validated results against the analytical solution

This pathway is ideal for small assemblies. For larger systems, see:
- [Pathway 3: FOM Concatenation](pathway3_fom_concatenation.ipynb)
- [Pathway 4: ROM Concatenation](pathway4_rom_concatenation.ipynb)